# Expérience 05 — Réglage de Kernel Ridge

## 1. Kernel Ridge, degré 2

In [1]:
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from sklearn.compose import TransformedTargetRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline

from src.evaluation import cv_rmse
from src.training import (
    MEMBERS, RANDOM_STATE, cross_validate_model, fit_model, load_data, make_model,
    make_preprocessing,
)

def show_title(title):
    display(Markdown(f"**{title}**"))

def show_table(table, float_format="{:.4f}".format):
    """Afficher un tableau sans index, nombres à 4 décimales, texte jamais coupé."""
    display(HTML(table.to_html(index=False, float_format=float_format)))

# =======================================================================
# Données et plis (mêmes plis que le notebook 05)
# =======================================================================
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
X, y, _ = load_data(root / "data/train.csv", root / "data/test.csv")
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.6, 1.0, 2.0, 5.0]
COEF0S = [0.5, 1.0, 2.0, 2.5, 3.0, 5.0, 10.0]
CURRENT = {"alpha": 0.6, "coef0": 2.5}
FOLDS = [f"Pli {k}" for k in range(1, folds.get_n_splits() + 1)]
SCORE = "RMSE log (moyenne des plis)"
SPREAD = "Écart type entre plis"

def make_kernel_ridge(alpha, coef0, degree):
    """Kernel Ridge polynomial avec le prétraitement de sa famille, sur log(SalePrice)."""
    return TransformedTargetRegressor(
        make_pipeline(make_preprocessing("Kernel Ridge"),
                      KernelRidge(alpha=alpha, kernel="polynomial", degree=degree, coef0=coef0)),
        func=np.log, inverse_func=np.exp)

def grid_search(degree):
    """Validation croisée (cross_validate_model) de chaque couple (alpha, coef0), triée par RMSE moyenne."""
    rows = []
    for alpha, coef0 in product(ALPHAS, COEF0S):
        _, scores = cross_validate_model(make_kernel_ridge(alpha, coef0, degree), X, y, folds)
        rows.append({"alpha": alpha, "coef0": coef0, **dict(zip(FOLDS, scores)),
                     SCORE: scores.mean(), SPREAD: scores.std(ddof=0)})
    table = pd.DataFrame(rows).sort_values(SCORE, kind="stable").reset_index(drop=True)
    table.insert(0, "Rang", table.index + 1)
    return table

def is_current(table):
    return table["alpha"].eq(CURRENT["alpha"]) & table["coef0"].eq(CURRENT["coef0"])

# =======================================================================
# 1.1 Grille alpha × coef0, degré 2
# =======================================================================
start = time.perf_counter()
grid_2 = grid_search(degree=2)
elapsed_2 = time.perf_counter() - start

show_title(f"1.1 Les 15 meilleures combinaisons sur {len(grid_2)} (degré 2)")
top = grid_2.head(15)
if not is_current(top).any():
    top = pd.concat([top, grid_2[is_current(grid_2)]])
show_table(top)

# =======================================================================
# 1.2 Meilleure combinaison et configuration actuelle
# =======================================================================
best = grid_2.iloc[0]
current = grid_2[is_current(grid_2)].iloc[0]
on_edge = {
    "alpha": best["alpha"] in (min(ALPHAS), max(ALPHAS)),
    "coef0": best["coef0"] in (min(COEF0S), max(COEF0S)),
}
show_title("1.2 Meilleure combinaison et configuration actuelle")
show_table(pd.DataFrame({
    "Mesure": [
        "Meilleure combinaison (alpha, coef0)",
        "Configuration actuelle (alpha, coef0)",
        "Rang de la configuration actuelle",
        "(a) RMSE moyenne : meilleure − actuelle",
        "(b) Plis où la meilleure fait mieux que l'actuelle",
        "(c) Meilleure au bord de la grille : alpha",
        "(c) Meilleure au bord de la grille : coef0",
    ],
    "Valeur": [
        f"({best['alpha']:g}, {best['coef0']:g})",
        f"({current['alpha']:g}, {current['coef0']:g})",
        f"{int(current['Rang'])} sur {len(grid_2)}",
        f"{best[SCORE] - current[SCORE]:+.4f}",
        f"{int((best[FOLDS].to_numpy() < current[FOLDS].to_numpy()).sum())} sur {len(FOLDS)}",
        "oui" if on_edge["alpha"] else "non",
        "oui" if on_edge["coef0"] else "non",
    ],
}))
print(f"Temps de la grille de degré 2 : {elapsed_2:.0f} s")

**1.1 Les 15 meilleures combinaisons sur 56 (degré 2)**

Rang,alpha,coef0,Pli 1,Pli 2,Pli 3,Pli 4,Pli 5,RMSE log (moyenne des plis),Écart type entre plis
1,0.6000,3.0000,0.1234,0.1088,0.1056,0.1080,0.1042,0.1100,0.0069
2,1.0000,5.0000,0.1241,0.1075,0.1052,0.1087,0.1045,0.1100,0.0072
3,0.6000,2.5000,0.1230,0.1089,0.1058,0.1081,0.1044,0.1100,0.0067
4,0.6000,5.0000,0.1240,0.1092,0.1053,0.1078,0.1043,0.1101,0.0072
5,1.0000,3.0000,0.1234,0.1076,0.1059,0.1092,0.1048,0.1102,0.0068
6,1.0000,10.0000,0.1244,0.1086,0.1050,0.1084,0.1050,0.1103,0.0072
7,0.6000,2.0000,0.1225,0.1094,0.1064,0.1083,0.1048,0.1103,0.0063
8,2.0000,10.0000,0.1250,0.1066,0.1053,0.1099,0.1053,0.1104,0.0075
9,1.0000,2.5000,0.1230,0.1081,0.1063,0.1095,0.1052,0.1104,0.0065
10,0.6000,10.0000,0.1240,0.1108,0.1057,0.1078,0.1049,0.1106,0.0070


**1.2 Meilleure combinaison et configuration actuelle**

Mesure,Valeur
"Meilleure combinaison (alpha, coef0)","(0.6, 3)"
"Configuration actuelle (alpha, coef0)","(0.6, 2.5)"
Rang de la configuration actuelle,3 sur 56
(a) RMSE moyenne : meilleure − actuelle,-0.0001
(b) Plis où la meilleure fait mieux que l'actuelle,4 sur 5
(c) Meilleure au bord de la grille : alpha,non
(c) Meilleure au bord de la grille : coef0,non


Temps de la grille de degré 2 : 61 s


## 2. Kernel Ridge, degré 3

In [2]:
# =======================================================================
# 2.1 Même grille, degré 3 (pour information)
# =======================================================================
start = time.perf_counter()
grid_3 = grid_search(degree=3)
elapsed_3 = time.perf_counter() - start

show_title(f"2.1 Les 5 meilleures combinaisons sur {len(grid_3)} (degré 3)")
show_table(grid_3.head(5))
print(f"Temps de la grille de degré 3 : {elapsed_3:.0f} s")

**2.1 Les 5 meilleures combinaisons sur 56 (degré 3)**

Rang,alpha,coef0,Pli 1,Pli 2,Pli 3,Pli 4,Pli 5,RMSE log (moyenne des plis),Écart type entre plis
1,5.0000,3.0000,0.1223,0.1102,0.1068,0.1084,0.1046,0.1104,0.0062
2,5.0000,5.0000,0.1237,0.1115,0.1063,0.1075,0.1038,0.1105,0.0070
3,2.0000,2.5000,0.1225,0.1127,0.1076,0.1076,0.1043,0.1109,0.0064
4,2.0000,3.0000,0.1232,0.1133,0.1076,0.1076,0.1040,0.1111,0.0067
5,5.0000,2.5000,0.1220,0.1112,0.1081,0.1096,0.1061,0.1114,0.0056


Temps de la grille de degré 3 : 67 s


## 3. Vérification de Lasso

In [3]:
# =======================================================================
# 3.1 α choisi par LassoCV dans chaque pli
# =======================================================================
low, high = min(MEMBERS["Lasso"].alphas), max(MEMBERS["Lasso"].alphas)
rows = []
for k, (fit_index, _) in enumerate(folds.split(X), start=1):
    model = fit_model(make_model(["Lasso"]), X.iloc[fit_index], y.iloc[fit_index])
    alpha = model.regressor_[-1].alpha_
    rows.append({"Pli": k, "α choisi": alpha, "Borne min": low, "Borne max": high,
                 "Égal à une borne": "oui" if np.isclose(alpha, low) or np.isclose(alpha, high) else "non"})

show_title("3.1 α choisi par LassoCV dans chaque pli")
show_table(pd.DataFrame(rows), float_format="{:.3e}".format)

**3.1 α choisi par LassoCV dans chaque pli**

Pli,α choisi,Borne min,Borne max,Égal à une borne
1,6.422e-04,3.162e-05,3.162e-02,non
2,7.667e-04,3.162e-05,3.162e-02,non
3,7.667e-04,3.162e-05,3.162e-02,non
4,6.422e-04,3.162e-05,3.162e-02,non
5,6.422e-04,3.162e-05,3.162e-02,non


## Bilan de l'expérience

**Objectif.** Les paramètres de Kernel Ridge (`alpha = 0,6`, `coef0 = 2,5`, degré 2) venaient de valeurs usuelles pour ce concours et n'avaient jamais été vérifiés. Cette expérience vérifie s'il existe de meilleurs réglages. Elle vérifie aussi que la liste de valeurs de α donnée à Lasso est assez large.

**Méthode.** Nous utilisons les mêmes cinq plis et le même prétraitement que le notebook 05, et nous mesurons la RMSE log sur les ventes typiques. Nous avons fixé la règle de décision avant de voir les résultats : nous changeons les paramètres de Kernel Ridge seulement si la meilleure combinaison gagne au moins 0,001 en moyenne et fait mieux dans au moins 4 plis sur 5. Un gain plus petit pourrait venir du hasard de la répartition en plis.

**Kernel Ridge, degré 2.** Nous avons essayé 56 combinaisons : 8 valeurs d'`alpha` (de 0,05 à 5) et 7 valeurs de `coef0` (de 0,5 à 10). Notre configuration arrive au 3ᵉ rang, à égalité avec la meilleure à quatre décimales (0,1100). La meilleure combinaison (`alpha = 0,6`, `coef0 = 3,0`) ne gagne que 0,0001, et son score varie même un peu plus d'un pli à l'autre (écart type de 0,0069 contre 0,0067). Les neuf meilleures combinaisons tiennent entre 0,1100 et 0,1104 : autour de nos réglages, le score change très peu. La meilleure combinaison n'est pas au bord de la grille, il n'est donc pas nécessaire de l'élargir.

**Kernel Ridge, degré 3.** Le degré 2 combine les caractéristiques deux par deux; le degré 3 les combinerait aussi trois par trois. Avec la même grille, la meilleure combinaison de degré 3 obtient 0,1104, contre 0,1100 pour le degré 2. Ajouter les produits de trois caractéristiques n'améliore donc pas la prédiction.

**Lasso.** LassoCV choisit lui-même son α par validation croisée, mais seulement parmi les 40 valeurs qu'on lui donne, de 3,2 × 10⁻⁵ à 3,2 × 10⁻². Si le α choisi était à l'une des deux bornes, la meilleure valeur pourrait se trouver en dehors de la liste. Dans chaque pli, LassoCV choisit un α entre 6,4 × 10⁻⁴ et 7,7 × 10⁻⁴, loin des deux bornes : la liste est assez large, et le choix est stable d'un pli à l'autre.

**Conséquence.** Selon la règle fixée d'avance, nous gardons les paramètres actuels de Kernel Ridge : `alpha = 0,6`, `coef0 = 2,5` et degré 2. La liste de valeurs de Lasso reste aussi la même. Aucun changement n'est apporté au modèle du notebook 05.